# scBriX: Bridging Alignment for Cross-Modal Multi-Omics Generation

We propose a cross-modal generative framework built around a transferable, **RNA-centered Bridging Alignment** strategy. Leveraging the richer and more comprehensive information content of scRNA-seq, RNA serves as the "semantic center" that connects other omic modalities. The framework is trained in progressive stages.


In [1]:
import sys
sys.path.append('..')
from src.scBriX import scBriX
import numpy as np
import torch
import scanpy as sc
from src.util import five_fold_split_dataset,cluster_metrics

d:\App\Anaconda\Lib\site-packages\scanpy\_utils\__init__.py:35: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  from anndata import __version__ as anndata_version
d:\App\Anaconda\Lib\site-packages\scanpy\__init__.py:24: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):
d:\App\Anaconda\Lib\site-packages\scanpy\readwrite.py:15: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):


In [ ]:
tdat = sc.read_h5ad('../data/BMMC/ADT_2.h5ad')
rdat1 = sc.read_h5ad('../data/BMMC/GEX_c1.h5ad')
rdat2 = sc.read_h5ad('../data/BMMC/GEX_c2.h5ad')
adat = sc.read_h5ad('../data/BMMC/ATAC_1.h5ad')
id_list = five_fold_split_dataset(rdat1)
train_id1, validation_id1, test_id1 = id_list[4]
train_r1, train_a= rdat1[train_id1+validation_id1,:], adat[train_id1+validation_id1,:]
id_list = five_fold_split_dataset(rdat2)
train_id2, validation_id2, test_id2 = id_list[4]
train_r2, train_t= rdat2[train_id2+validation_id2,:], tdat[train_id2+validation_id2,:]

In [3]:
from src.config import cfg
device = 'cuda' if torch.cuda.is_available() else 'cpu'

## Stage 1: Joint RNA–ATAC Training

Using paired scRNA-seq and scATAC-seq data, we jointly train RNA and ATAC conditional encoders. In the aligned space, contrastive learning together with a DEC-style clustering objective maximizes the mutual information between representations of the same cell across modalities, thereby establishing a core feature space.

In [19]:

step1 = ["atac","rna"]
cfg_rna_atac = {xtype: cfg[xtype] for xtype in step1}
scBriX_rna_atac = scBriX(device=device, cfg=cfg_rna_atac, xtypes=step1, n_clusters=22,save_path='../param/brid').to(device)

scBriX_rna_atac.svd({"rna":train_r1,"atac":train_a})

Saved cell_type labels for rna modality
Created features and saved for rna
Created features and saved for atac


In [21]:

scBriX_rna_atac.lr =3e-5
scBriX_rna_atac.phases = {
                'pretrain': {'contrast': 0.0, "intra":0.0,"dec":0},
                'warmup': {'contrast': 0.05, "intra": 0.5, "dec": 0},
                'full': {'contrast': 0.3, "intra": 1,"dec": 0.4}}
scBriX_rna_atac.train_model(n_epoch=900)

 74%|███████▍  | 666/900 [4:53:20<1:43:04, 26.43s/it, Phase: full | Loss: 1.1395 | contrastLoss: 0.6531 | intraLoss: 0.0066 | DECLoss: 1.0142 | atac: 0.0042 | rna: 0.1175 | contrastWeight: 0.201 | intraWeight: 0.802 | decWeight: 0.242]    


Early stopping at epoch 666


## Stage 2: RNA-Anchored ADT Alignment

We freeze the RNA conditional encoder trained in Stage 1 as a fixed semantic anchor. We then introduce a new modality — antibody-derived tags (ADT) — and train only the ADT conditional encoder with paired RNA–ADT data. Through contrastive learning against the frozen RNA encoder, ADT embeddings are aligned to the same RNA-centered semantic space.

In [6]:
step2 = ["rna","adt"]
cfg_rna_adt = {xtype: cfg[xtype] for xtype in step2}
scBriX_rna_adt = scBriX(device=device, cfg=cfg_rna_adt, xtypes=step2,n_clusters=45,save_path='../param/brid').to(device)
scBriX_rna_adt.svd({"rna":train_r2,"adt":train_t})

Created features and saved for rna
Saved cell_type labels for adt modality
Created features and saved for adt


In [7]:
scBriX_rna_adt.load_model(param_dict={"rna":"../param/brid/model_rna.pth"},freeze=True)
scBriX_rna_adt.lr = 4e-5
scBriX_rna_adt.phases = {
                'pretrain': {'contrast': 0.0, "intra":0.0,"dec":0},
                'warmup': {'contrast': 0.1, "intra": 0.5, "dec": 0},
                'full': {'contrast': 0.3, "intra": 1,"dec": 0.4}}
scBriX_rna_adt.train_model(n_epoch=900)

100%|██████████| 900/900 [8:22:16<00:00, 33.48s/it, Phase: full | Loss: 2.1121 | contrastLoss: 0.8819 | intraLoss: 0.0043 | DECLoss: 1.2297 | rna: 0.1417 | adt: 0.1321 | contrastWeight: 0.300 | intraWeight: 1.000 | decWeight: 0.400]      


## Cross-Modal Generation

With both ADT and ATAC encoders aligned to the same RNA-centered semantic space, we can perform cross-modal generation (e.g., ADT→ATAC and ATAC→ADT) without requiring direct ADT–ATAC paired data during training. This design reduces reliance on fully paired datasets and scales efficiently to future modalities.

In [ ]:
step3 = ["atac","adt"]

scBriX = scBriX(device=device, cfg=cfg, xtypes=step3,save_path='../param/brid').to(device)
scBriX.load_model(param_dict={"adt":"../param/brid/model_adt.pth","atac":"../param/brid/model_atac.pth"},freeze=True)

In [15]:
p_atac = scBriX.inference(out_types=["atac"],condition_data={"adt":tdat[test_id2,:]})["atac"]



d:\Lab\code\scBriX\tutorial\..\src\sampler.py:32: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  gt = torch.tensor(gt).to(device)
time: 0: 100%|██████████| 1000/1000 [09:21<00:00,  1.78it/s] 


In [16]:

sc.pp.pca(p_atac)
sc.pp.neighbors(p_atac)
sc.tl.tsne(p_atac)
sc.tl.leiden(p_atac)
result = cluster_metrics(p_atac)
print('ADT->ATAC generation:\nARI: %.3f, \tNMI: %.3f' % (result["ARI"], result["NMI"]))

d:\App\Anaconda\Lib\site-packages\scanpy\preprocessing\_pca\__init__.py:245: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  Version(ad.__version__) < Version("0.9")
d:\App\Anaconda\Lib\site-packages\scanpy\neighbors\__init__.py:427: FutureWarning: Use obsm (e.g. `k in adata.obsm` or `adata.obsm.keys() | {'u'}`) instead of AnnData.obsm_keys, AnnData.obsm_keys is deprecated and will be removed in the future.
  if "X_diffmap" in adata.obsm_keys():


ADT->ATAC generation:
ARI: 0.511, 	NMI: 0.579


In [17]:
p_adt = scBriX.inference(out_types=["adt"],condition_data={"atac":adat[test_id1,:]})["adt"]

['atac'] pretrained


d:\Lab\code\scBriX\tutorial\..\src\sampler.py:32: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  gt = torch.tensor(gt).to(device)
time: 0: 100%|██████████| 1000/1000 [07:36<00:00,  2.19it/s] 


In [18]:
sc.pp.pca(p_adt)
sc.pp.neighbors(p_adt)
sc.tl.tsne(p_adt)
sc.tl.leiden(p_adt)
result = cluster_metrics(p_adt)
print('ATAC->ADT generation:\nARI: %.3f, \tNMI: %.3f' % (result["ARI"], result["NMI"]))

d:\App\Anaconda\Lib\site-packages\scanpy\preprocessing\_pca\__init__.py:245: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  Version(ad.__version__) < Version("0.9")
d:\App\Anaconda\Lib\site-packages\scanpy\neighbors\__init__.py:427: FutureWarning: Use obsm (e.g. `k in adata.obsm` or `adata.obsm.keys() | {'u'}`) instead of AnnData.obsm_keys, AnnData.obsm_keys is deprecated and will be removed in the future.
  if "X_diffmap" in adata.obsm_keys():


ATAC->ADT generation:
ARI: 0.341, 	NMI: 0.568
